# GNN 사기 탐지 — 05-1. RGCN(Baseline) vs PC-GNN

## 설계 결정 요약

| 항목 | RGCN (베이스라인) | PC-GNN |
|------|------------------|--------|
| 피처 | rating_norm 1차원 | 행동피처 8개 + TF-IDF SVD 64차원 |
| 엣지 | **R-U-R (1종만)** | R-U-R + R-S-R + R-P-R + R-Sim-R (4종) |
| 집계 | 관계별 균등 평균 (RGCNConv) | Chooser 어텐션 (위장 이웃 억제) |
| 배치 | 전체 train | **Picker** — 사기 전량 + 정상 3배 |
| 손실 | CrossEntropyLoss + class weight | **FocalLoss** + class weight |
| 분할 | **유저 단위** (R-U-R 라벨 누수 차단) | 동일 |

### 비교 포인트
- RGCN: rating_norm 1차원 + R-U-R만 → 유저 단위 분할로 val/test 노드가 train과 단절 → 그래프 없는 수준의 기준선
- PC-GNN: EDA 72차원 피처 + 4종 엣지(R-S-R·R-P-R·R-Sim-R 포함) + Chooser + Picker → 세 가지 개선의 결합 효과


In [14]:
# [00] 임포트 및 환경 설정
import os, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score, average_precision_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch_geometric.nn import MessagePassing, RGCNConv
from torch_geometric.utils import to_undirected

BASE_DIR = r'C:\Users\bella\Desktop\대학\CODE\GNN 학술제'
os.chdir(BASE_DIR)
torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ 환경 설정 완료 | 디바이스: {device}')
print(f'   PyTorch: {torch.__version__}')

✅ 환경 설정 완료 | 디바이스: cpu
   PyTorch: 2.11.0+cpu


In [15]:
# [01] 하이퍼파라미터 설정
CFG = {
    'random_state': 42,
    # 텍스트 임베딩
    'tfidf_max_features': 5000,
    'svd_n_components':   64,
    # 유저 단위 분할 비율
    'train_user_ratio': 0.60,
    'val_user_ratio':   0.20,   # test = 나머지 0.20
    # 엣지 샘플링 — 상한 설정으로 heavy user 폭발 방지
    'rur_max_per_node':  10,    # R-U-R: 노드당 최대 10개 이웃 (CPU 속도 최적화)
    'rsr_max_per_node':   5,
    'rpr_max_per_node':   8,
    'rsim_thr':           0.70,
    # 모델 — hidden 64로 어텐션 연산 4× 감소
    'hidden_dim':  64,
    'dropout':     0.40,
    # 학습
    'lr':       5e-4,
    'wd':       5e-4,
    'epochs':   60,
    'patience': 10,
    # PC-GNN 특화
    'picker_ratio': 3,    # 배치 내 정상:사기 = 3:1
    'focal_gamma':  2.0,
}
print('✅ 설정 완료')
print(f"   유저 분할: train {CFG['train_user_ratio']*100:.0f}% / val {CFG['val_user_ratio']*100:.0f}% / test 나머지")
print(f"   Picker ratio={CFG['picker_ratio']}  |  Focal gamma={CFG['focal_gamma']}")
print(f"   hidden_dim={CFG['hidden_dim']}  |  R-U-R max_per_node={CFG['rur_max_per_node']}  |  epochs={CFG['epochs']}")

✅ 설정 완료
   유저 분할: train 60% / val 20% / test 나머지
   Picker ratio=3  |  Focal gamma=2.0
   hidden_dim=64  |  R-U-R max_per_node=10  |  epochs=60


In [16]:
# [02] 데이터 로드
print('데이터 로드 중...')
df = pd.read_parquet('yelpzip_sampled.parquet')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

assert set(df['label'].unique()).issubset({0, 1}), '라벨 변환 필요!'
N = len(df)
labels = df['label'].values
n_fraud = labels.sum()
print(f'✅ 데이터: {N:,}건  | 사기: {n_fraud:,} ({100*n_fraud/N:.1f}%)  | 정상: {N-n_fraud:,}')
print(f'   기간: {df["date"].min().date()} ~ {df["date"].max().date()}')
print(f'   유저: {df["user_id"].nunique():,}명  | 식당: {df["prod_id"].nunique():,}개')

데이터 로드 중...
✅ 데이터: 50,950건  | 사기: 12,659 (24.8%)  | 정상: 38,291
   기간: 2013-01-01 ~ 2014-12-31
   유저: 17,439명  | 식당: 200개


In [17]:
# [03] TF-IDF + SVD 텍스트 임베딩 (임시 train 인덱스로 fit)
_users = df['user_id'].unique().copy()
np.random.seed(CFG['random_state'])
np.random.shuffle(_users)
_n_tr     = int(len(_users) * CFG['train_user_ratio'])
_fit_idx  = df.index[df['user_id'].isin(set(_users[:_n_tr]))].to_numpy()

print('TF-IDF + SVD 임베딩 중...')
t0 = time.time()
tfidf = TfidfVectorizer(
    max_features=CFG['tfidf_max_features'], ngram_range=(1,2),
    min_df=3, max_df=0.95, strip_accents='unicode', sublinear_tf=True
)
tfidf.fit(df['text'].astype(str).values[_fit_idx])
tfidf_mat  = tfidf.transform(df['text'].astype(str))

svd = TruncatedSVD(n_components=CFG['svd_n_components'], random_state=CFG['random_state'])
svd.fit(tfidf_mat[_fit_idx])
text_embed = svd.transform(tfidf_mat).astype(np.float32)
print(f'✅ 임베딩 완료 ({time.time()-t0:.0f}s) | 설명 분산: {svd.explained_variance_ratio_.sum():.3f}')

TF-IDF + SVD 임베딩 중...
✅ 임베딩 완료 (17s) | 설명 분산: 0.128


In [18]:
# [04] 유저 단위 Train / Val / Test 분리
# R-U-R 라벨 누수 차단: 동일 유저 모든 리뷰가 같은 파티션에 배치됨
users = df['user_id'].unique()
np.random.seed(CFG['random_state'])
np.random.shuffle(users)
n_users = len(users)

n_train = int(n_users * CFG['train_user_ratio'])
n_val   = int(n_users * CFG['val_user_ratio'])

train_users = set(users[:n_train])
val_users   = set(users[n_train:n_train+n_val])
test_users  = set(users[n_train+n_val:])

idx_train = df.index[df['user_id'].isin(train_users)].to_numpy()
idx_val   = df.index[df['user_id'].isin(val_users)].to_numpy()
idx_test  = df.index[df['user_id'].isin(test_users)].to_numpy()

train_mask = torch.zeros(N, dtype=torch.bool); train_mask[idx_train] = True
val_mask   = torch.zeros(N, dtype=torch.bool); val_mask[idx_val]     = True
test_mask  = torch.zeros(N, dtype=torch.bool); test_mask[idx_test]   = True

df_train = df.iloc[idx_train]
print(f'Train: {len(idx_train):,}건  사기율: {labels[idx_train].mean():.3f}  ({len(train_users):,}명 유저)')
print(f'Val  : {len(idx_val):,}건  사기율: {labels[idx_val].mean():.3f}  ({len(val_users):,}명 유저)')
print(f'Test : {len(idx_test):,}건  사기율: {labels[idx_test].mean():.3f}  ({len(test_users):,}명 유저)')
print('✅ 유저 단위 분리 완료 — train/val/test 간 R-U-R 엣지 없음')

Train: 30,620건  사기율: 0.246  (10,463명 유저)
Val  : 9,912건  사기율: 0.260  (3,487명 유저)
Test : 10,418건  사기율: 0.243  (3,489명 유저)
✅ 유저 단위 분리 완료 — train/val/test 간 R-U-R 엣지 없음


In [19]:
# [05] 사용자 통계 피처 (Train 전용 → 누수 방지)
print('사용자 통계 계산 중...')
user_stats = df_train.groupby('user_id').agg(
    user_review_count = ('label', 'count'),
    user_avg_rating   = ('rating', 'mean'),
    user_rating_std   = ('rating', lambda x: float(x.std()) if len(x)>1 else 0.0),
    activity_span     = ('date',   lambda x: float((x.max()-x.min()).days)),
).reset_index()
user_stats.fillna(0, inplace=True)

burst_df = pd.read_csv('burst_features.csv', index_col=0).reset_index()
burst_df.columns = ['user_id','user_burst_max7d','user_burst_ratio']
user_stats = user_stats.merge(burst_df, on='user_id', how='left')
user_stats[['user_burst_max7d','user_burst_ratio']] = user_stats[['user_burst_max7d','user_burst_ratio']].fillna(0)
print(f'✅ 사용자 통계 완료: {len(user_stats):,}명')

사용자 통계 계산 중...
✅ 사용자 통계 완료: 10,463명


In [20]:
# [06] 피처 행렬 조립
df_feat = df[['user_id','prod_id','rating','date','label']].copy().reset_index(drop=True)

user_cols = ['user_id','user_review_count','user_avg_rating','user_rating_std',
             'activity_span','user_burst_max7d','user_burst_ratio']
df_feat = df_feat.merge(user_stats[user_cols], on='user_id', how='left')

fr = pd.read_csv('features_rating.csv', index_col=0)
fr_dedup = fr[['user_id','prod_id','rating_deviation','is_cold_start','is_single_review']].drop_duplicates(subset=['user_id','prod_id'])
df_feat = df_feat.merge(fr_dedup, on=['user_id','prod_id'], how='left')
df_feat.fillna(0, inplace=True)
df_feat['user_avg_rating'] = df_feat['user_avg_rating'].replace(0, df_train['rating'].mean())

# rpr_suspicion — R-P-R 엣지 및 노드 피처용
rating_dev = df_feat['rating_deviation'].fillna(0).values
df_feat['rpr_suspicion'] = np.abs(rating_dev)
# 장기 충성 고객 화이트리스트
whitelist = (df_feat['activity_span'].fillna(0) >= 365) & (df_feat['user_review_count'].fillna(0) >= 10)
df_feat.loc[whitelist, 'rpr_suspicion'] = 0.0

# ── RGCN 베이스라인: rating_norm 1차원 ──
X_rgcn = (df_feat['rating'].values / 5.0).reshape(-1,1).astype(np.float32)

# ── PC-GNN: 행동 피처 8개 + TF-IDF SVD 64차원 ──
BEHAV_COLS = [
    'rating', 'user_review_count', 'user_avg_rating', 'user_rating_std',
    'activity_span', 'user_burst_max7d', 'is_cold_start', 'is_single_review',
]
scaler   = StandardScaler()
scaler.fit(df_feat[BEHAV_COLS].values[idx_train])
behav_mat = scaler.transform(df_feat[BEHAV_COLS].values).astype(np.float32)
X_pcgnn   = np.hstack([behav_mat, text_embed])
X_pcgnn   = np.nan_to_num(X_pcgnn, nan=0.0, posinf=0.0, neginf=0.0)

x_rgcn   = torch.tensor(X_rgcn,  dtype=torch.float32)
x_pcgnn  = torch.tensor(X_pcgnn, dtype=torch.float32)
y        = torch.tensor(labels,   dtype=torch.long)
in_dim_rgcn  = x_rgcn.shape[1]
in_dim_pcgnn = x_pcgnn.shape[1]

print('✅ 피처 조립 완료')
print(f'   RGCN  피처: {in_dim_rgcn}차원  (rating_norm)')
print(f'   PCGNN 피처: {in_dim_pcgnn}차원  (행동 {len(BEHAV_COLS)}개 + SVD {CFG["svd_n_components"]}차원)')

✅ 피처 조립 완료
   RGCN  피처: 1차원  (rating_norm)
   PCGNN 피처: 72차원  (행동 8개 + SVD 64차원)


In [21]:
# [07] 엣지 구성
df['ym'] = df['date'].dt.to_period('M')
rng = np.random.default_rng(42)

susp_scores       = df_feat['rpr_suspicion'].values
rating_dev_scores = np.abs(df_feat['rating_deviation'].fillna(0).values)

def make_pairs(groups, max_per_node=None):
    src, dst = [], []
    for nodes in groups:
        nodes = list(nodes)
        if len(nodes) < 2: continue
        n_pairs = len(nodes)*(len(nodes)-1)//2
        if max_per_node is None or n_pairs <= max_per_node*len(nodes):
            for i in range(len(nodes)):
                for j in range(i+1, len(nodes)):
                    src.append(nodes[i]); dst.append(nodes[j])
        else:
            for node in nodes:
                others = [n for n in nodes if n != node]
                k = min(max_per_node, len(others))
                for j in rng.choice(others, k, replace=False):
                    src.append(node); dst.append(int(j))
    return np.array(src, dtype=np.int64), np.array(dst, dtype=np.int64)

def make_pairs_scored(groups, scores, max_per_node):
    src, dst = [], []
    for nodes in groups:
        nodes = list(nodes)
        if len(nodes) < 2: continue
        n_pairs = len(nodes)*(len(nodes)-1)//2
        if n_pairs <= max_per_node*len(nodes):
            for i in range(len(nodes)):
                for j in range(i+1, len(nodes)):
                    src.append(nodes[i]); dst.append(nodes[j])
        else:
            sorted_nodes = [nodes[i] for i in np.argsort(-scores[nodes])]
            for node in nodes:
                others = [n for n in sorted_nodes if n != node]
                k = min(max_per_node, len(others))
                for j in others[:k]:
                    src.append(node); dst.append(j)
    return np.array(src, dtype=np.int64), np.array(dst, dtype=np.int64)

def to_edge(s, d, n):
    if len(s) == 0: return torch.zeros((2,0), dtype=torch.long)
    ei = torch.tensor(np.stack([s,d], axis=0), dtype=torch.long)
    ei = to_undirected(ei, num_nodes=n)
    return ei[:, ei[0] != ei[1]]

t0 = time.time()
print(f'엣지 구성 중... (N={N:,})')

# ── PC-GNN용 R-U-R: max_per_node 제한으로 heavy user 엣지 폭발 방지 ──
s, d = make_pairs(
    df.groupby('user_id').apply(lambda x: x.index.tolist()).values,
    max_per_node=CFG['rur_max_per_node'])
edge_rur = to_edge(s, d, N)
print(f'  R-U-R(전체,capped) : {edge_rur.size(1):>8,}개  (PC-GNN | 노드당 최대 {CFG["rur_max_per_node"]}개)')

# ── RGCN 베이스라인 전용: train 노드들 사이에서만 R-U-R 구성 ──
# val/test 노드가 완전히 고립됨 → GNN이 rating_norm 1차원 피처만으로 판단
df_tr = df.iloc[idx_train]
s_tr, d_tr = make_pairs(
    df_tr.groupby('user_id').apply(lambda x: x.index.tolist()).values,
    max_per_node=CFG['rur_max_per_node'])
edge_rur_train = to_edge(s_tr, d_tr, N)
print(f'  R-U-R(train,capped) : {edge_rur_train.size(1):>8,}개  (RGCN 베이스라인 전용)')

# R-S-R: PC-GNN 전용 — rating_deviation 높은 순 우선
s, d = make_pairs_scored(
    df.groupby(['prod_id','rating']).apply(lambda x: x.index.tolist()).values,
    rating_dev_scores, max_per_node=CFG['rsr_max_per_node'])
edge_rsr = to_edge(s, d, N)
print(f'  R-S-R               : {edge_rsr.size(1):>8,}개  (PC-GNN | rating_deviation 우선)')

# R-P-R: PC-GNN 전용 — rpr_suspicion 높은 순 우선
susp_mask = df_feat['rpr_suspicion'].values > 0
df_susp   = df[susp_mask].copy()
s, d = make_pairs_scored(
    df_susp.groupby('prod_id').apply(lambda x: x.index.tolist()).values,
    susp_scores, max_per_node=CFG['rpr_max_per_node'])
edge_rpr = to_edge(s, d, N)
print(f'  R-P-R               : {edge_rpr.size(1):>8,}개  (PC-GNN | 의심도 우선)')

# R-Sim-R: PC-GNN 전용 — 텍스트 유사도
sim_df = pd.read_csv('custom_edges_textSim.csv')
sim_df = sim_df[sim_df['text_similarity'] >= CFG['rsim_thr']]
valid  = (sim_df['src_review_idx'] < N) & (sim_df['dst_review_idx'] < N)
sim_df = sim_df[valid]
edge_rsim = to_edge(
    sim_df['src_review_idx'].values.astype(np.int64),
    sim_df['dst_review_idx'].values.astype(np.int64), N)
print(f'  R-Sim-R             : {edge_rsim.size(1):>8,}개  (PC-GNN | 유사도 ≥{CFG["rsim_thr"]})')

# ── RGCN 베이스라인: train-only R-U-R, 단일 관계 ──
edge_index_rgcn = edge_rur_train
edge_type_rgcn  = torch.zeros(edge_rur_train.size(1), dtype=torch.long)

# ── PC-GNN: 4개 관계 별도 유지 ──
edge_indices_pcgnn = [edge_rur, edge_rsr, edge_rpr, edge_rsim]
REL_NAMES = ['R-U-R','R-S-R','R-P-R','R-Sim-R']

print(f'\n✅ 엣지 구성 완료 ({time.time()-t0:.0f}s)')
print(f'   RGCN  엣지: {edge_index_rgcn.size(1):,}개  (train-only R-U-R → val/test 고립)')
print(f'   PCGNN 엣지: {sum(e.size(1) for e in edge_indices_pcgnn):,}개  (4종 분리)')

엣지 구성 중... (N=50,950)
  R-U-R(전체,capped) :  254,784개  (PC-GNN | 노드당 최대 10개)
  R-U-R(train,capped) :  150,180개  (RGCN 베이스라인 전용)
  R-S-R               :  481,794개  (PC-GNN | rating_deviation 우선)
  R-P-R               :  724,080개  (PC-GNN | 의심도 우선)
  R-Sim-R             :  319,670개  (PC-GNN | 유사도 ≥0.7)

✅ 엣지 구성 완료 (2s)
   RGCN  엣지: 150,180개  (train-only R-U-R → val/test 고립)
   PCGNN 엣지: 1,780,328개  (4종 분리)


In [22]:
# [08] 모델 정의

# ── Focal Loss ──────────────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


# ── Baseline RGCN ───────────────────────────────────────────────────
class BaselineRGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim=64, n_rel=2, n_class=2, dropout=0.3):
        super().__init__()
        self.conv1 = RGCNConv(in_dim,     hidden_dim, num_relations=n_rel)
        self.conv2 = RGCNConv(hidden_dim, hidden_dim, num_relations=n_rel)
        self.drop  = nn.Dropout(dropout)
        self.clf   = nn.Linear(hidden_dim, n_class)

    def forward(self, x, edge_index, edge_type):
        h = F.relu(self.conv1(x, edge_index, edge_type)); h = self.drop(h)
        h = F.relu(self.conv2(h, edge_index, edge_type)); h = self.drop(h)
        return self.clf(h)


# ── PC-GNN Chooser ──────────────────────────────────────────────────
class ChooserConv(MessagePassing):
    def __init__(self, in_ch, out_ch):
        super().__init__(aggr='add')
        self.lin  = nn.Linear(in_ch, out_ch, bias=False)
        self.attn = nn.Linear(2 * out_ch, 1)
        self.norm = nn.LayerNorm(out_ch)
        nn.init.xavier_uniform_(self.lin.weight)

    def forward(self, x, edge_index):
        if edge_index.size(1) == 0:
            return self.norm(torch.zeros(x.size(0), self.lin.weight.size(0), device=x.device))
        x_lin = self.lin(x)
        out   = self.propagate(edge_index, x=x_lin)
        return self.norm(out)

    def message(self, x_i, x_j):
        alpha = torch.sigmoid(self.attn(torch.cat([x_i, x_j], dim=-1)))
        return alpha * x_j


# ── PC-GNN ──────────────────────────────────────────────────────────
class PCGNN(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, n_rel=4, n_class=2, dropout=0.3):
        super().__init__()
        self.input_proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.LayerNorm(hidden_dim), nn.ELU()
        )
        self.drop      = nn.Dropout(dropout)
        self.choosers1 = nn.ModuleList([ChooserConv(hidden_dim, hidden_dim) for _ in range(n_rel)])
        self.choosers2 = nn.ModuleList([ChooserConv(hidden_dim, hidden_dim) for _ in range(n_rel)])
        self.inter_attn = nn.Sequential(
            nn.Linear(hidden_dim, 32), nn.Tanh(), nn.Linear(32, 1, bias=False)
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64), nn.ELU(),
            nn.Dropout(dropout), nn.Linear(64, n_class)
        )

    def forward(self, x, edge_indices):
        h  = self.drop(self.input_proj(x))
        h1 = [F.elu(self.drop(c(h,   ei))) for c, ei in zip(self.choosers1, edge_indices)]
        h2 = [F.elu(self.drop(c(h1i, ei))) for c, h1i, ei in zip(self.choosers2, h1, edge_indices)]
        stk   = torch.stack(h2, dim=1)
        attn  = torch.softmax(self.inter_attn(stk).squeeze(-1), dim=1)
        h_agg = (stk * attn.unsqueeze(-1)).sum(dim=1)
        return self.classifier(h_agg), attn.detach().mean(dim=0)


print('✅ 모델 정의 완료  (FocalLoss + BaselineRGCN + PCGNN)')
_r = BaselineRGCN(in_dim_rgcn, 64); _p = PCGNN(in_dim_pcgnn, CFG['hidden_dim'], 4, dropout=CFG['dropout'])
print(f'   RGCN  파라미터: {sum(p.numel() for p in _r.parameters() if p.requires_grad):,}개')
print(f'   PCGNN 파라미터: {sum(p.numel() for p in _p.parameters() if p.requires_grad):,}개')
del _r, _p

✅ 모델 정의 완료  (FocalLoss + BaselineRGCN + PCGNN)
   RGCN  파라미터: 12,738개
   PCGNN 파라미터: 46,026개


In [23]:
# [09] 학습 유틸리티

def picker_sample(idx_train, labels, ratio, N):
    fraud_idx  = idx_train[labels[idx_train] == 1]
    normal_idx = idx_train[labels[idx_train] == 0]
    n_sample   = min(len(fraud_idx) * ratio, len(normal_idx))
    sampled    = np.random.choice(normal_idx, n_sample, replace=False)
    picked     = np.concatenate([fraud_idx, sampled])
    mask = torch.zeros(N, dtype=torch.bool)
    mask[picked] = True
    return mask

def eval_rgcn(model, x, ei, et, y, mask):
    model.eval()
    with torch.no_grad():
        logits = model(x, ei, et)
        probs  = torch.softmax(logits, dim=1)[:, 1]
        preds  = logits.argmax(dim=1)
    yt, yp, yprob = y[mask].numpy(), preds[mask].numpy(), probs[mask].numpy()
    return average_precision_score(yt, yprob), f1_score(yt, yp, average='macro'), yt, yp, yprob

def eval_pcgnn(model, x, edge_list, y, mask):
    model.eval()
    with torch.no_grad():
        logits, _ = model(x, edge_list)
        probs     = torch.softmax(logits, dim=1)[:, 1]
        preds     = logits.argmax(dim=1)
    yt, yp, yprob = y[mask].numpy(), preds[mask].numpy(), probs[mask].numpy()
    return average_precision_score(yt, yprob), f1_score(yt, yp, average='macro'), yt, yp, yprob

def bar(v, L=28): return '█'*int(v*L) + '░'*(L-int(v*L))

cw = compute_class_weight('balanced', classes=np.array([0,1]), y=labels[idx_train])
print('✅ 유틸리티 준비 완료')
print(f'   클래스 가중치: 정상={cw[0]:.3f}  사기={cw[1]:.3f}')
print(f'   Picker: 사기 전량 + 정상 {CFG["picker_ratio"]}배 → 배치 사기율 {1/(1+CFG["picker_ratio"]):.2f}')

✅ 유틸리티 준비 완료
   클래스 가중치: 정상=0.664  사기=2.029
   Picker: 사기 전량 + 정상 3배 → 배치 사기율 0.25


In [24]:
# [10] RGCN 베이스라인 학습
print('='*62)
print(' RGCN 베이스라인 학습')
print(' 피처: rating_norm (1차원)  |  집계: 균등 평균')
print(' 엣지: R-U-R (1종) — 유저 단위 분할로 val/test 단절')
print('='*62)

cw_r   = compute_class_weight('balanced', classes=np.array([0,1]), y=labels[idx_train])
crit_r = nn.CrossEntropyLoss(weight=torch.tensor(cw_r, dtype=torch.float32))

# n_rel=1: 단일 관계 타입 (R-U-R만)
rgcn_model = BaselineRGCN(in_dim=in_dim_rgcn, hidden_dim=64, n_rel=1, dropout=0.30)
opt_r = AdamW(rgcn_model.parameters(), lr=1e-3, weight_decay=1e-4)
sch_r = CosineAnnealingLR(opt_r, T_max=CFG['epochs'])

best_r_pr, best_r_state, no_imp = 0.0, None, 0
t0 = time.time()

for ep in range(1, CFG['epochs']+1):
    rgcn_model.train(); opt_r.zero_grad()
    logits = rgcn_model(x_rgcn, edge_index_rgcn, edge_type_rgcn)
    loss   = crit_r(logits[train_mask], y[train_mask])
    loss.backward(); nn.utils.clip_grad_norm_(rgcn_model.parameters(), 1.0); opt_r.step()
    sch_r.step()

    if ep % 10 == 0:
        vpr, vf1, *_ = eval_rgcn(rgcn_model, x_rgcn, edge_index_rgcn, edge_type_rgcn, y, val_mask)
        print(f'  [E{ep:3d}] loss={float(loss):.4f} | val PR-AUC={vpr:.4f} F1={vf1:.4f} ({time.time()-t0:.0f}s)')
        if vpr > best_r_pr:
            best_r_pr = vpr
            best_r_state = {k: v.clone() for k,v in rgcn_model.state_dict().items()}
            no_imp = 0
        else:
            no_imp += 1
            if no_imp >= CFG['patience'] // 2:
                print(f'  Early stopping'); break

rgcn_model.load_state_dict(best_r_state)
res_r_pr, res_r_f1, yt_r, yp_r, yprob_r = eval_rgcn(
    rgcn_model, x_rgcn, edge_index_rgcn, edge_type_rgcn, y, test_mask)
torch.save(best_r_state, 'best_rgcn_baseline.pt')
print(f'\n✅ RGCN 최종:  PR-AUC={res_r_pr:.4f}  Macro F1={res_r_f1:.4f}')


 RGCN 베이스라인 학습
 피처: rating_norm (1차원)  |  집계: 균등 평균
 엣지: R-U-R (1종) — 유저 단위 분할로 val/test 단절
  [E 10] loss=0.6558 | val PR-AUC=0.2921 F1=0.2065 (2s)
  [E 20] loss=0.6108 | val PR-AUC=0.2921 F1=0.2065 (4s)
  [E 30] loss=0.5787 | val PR-AUC=0.2921 F1=0.2065 (8s)
  [E 40] loss=0.5575 | val PR-AUC=0.2921 F1=0.2065 (11s)
  [E 50] loss=0.5472 | val PR-AUC=0.2921 F1=0.2065 (14s)
  [E 60] loss=0.5438 | val PR-AUC=0.2921 F1=0.2065 (17s)
  Early stopping

✅ RGCN 최종:  PR-AUC=0.2739  Macro F1=0.1956


In [25]:
# [11] PC-GNN 학습
print('='*62)
print(' PC-GNN 학습')
print(f' 피처: 행동 {len(BEHAV_COLS)}개 + SVD {CFG["svd_n_components"]}차원 = {in_dim_pcgnn}차원')
print(' 엣지: R-U-R + R-S-R + R-P-R + R-Sim-R (4종) | Chooser 어텐션')
print(f' 배치: Picker ratio={CFG["picker_ratio"]}  |  FocalLoss gamma={CFG["focal_gamma"]}')
print('='*62)

cw_p   = compute_class_weight('balanced', classes=np.array([0,1]), y=labels[idx_train])
cw_p[1] *= 1.5
crit_p = FocalLoss(alpha=torch.tensor(cw_p, dtype=torch.float32), gamma=CFG['focal_gamma'])

pcgnn_model = PCGNN(in_dim=in_dim_pcgnn, hidden_dim=CFG['hidden_dim'], n_rel=4, dropout=CFG['dropout'])
opt_p = AdamW(pcgnn_model.parameters(), lr=CFG['lr'], weight_decay=CFG['wd'])
sch_p = CosineAnnealingLR(opt_p, T_max=CFG['epochs'])

best_p_pr, best_p_state, no_imp_p = 0.0, None, 0
t0 = time.time()

for ep in range(1, CFG['epochs']+1):
    picked_mask = picker_sample(idx_train, labels, CFG['picker_ratio'], N)

    pcgnn_model.train(); opt_p.zero_grad()
    logits, _ = pcgnn_model(x_pcgnn, edge_indices_pcgnn)
    loss = crit_p(logits[picked_mask], y[picked_mask])
    loss.backward(); nn.utils.clip_grad_norm_(pcgnn_model.parameters(), 1.0); opt_p.step()
    sch_p.step()

    if ep % 10 == 0:
        vpr, vf1, *_ = eval_pcgnn(pcgnn_model, x_pcgnn, edge_indices_pcgnn, y, val_mask)
        print(f'  [E{ep:3d}] loss={float(loss):.4f} | val PR-AUC={vpr:.4f} F1={vf1:.4f} ({time.time()-t0:.0f}s)')
        if vpr > best_p_pr:
            best_p_pr = vpr
            best_p_state = {k: v.clone() for k,v in pcgnn_model.state_dict().items()}
            no_imp_p = 0
        else:
            no_imp_p += 1
            if no_imp_p >= CFG['patience']:
                print(f'  Early stopping'); break

pcgnn_model.load_state_dict(best_p_state)
res_p_pr, res_p_f1, yt_p, yp_p, yprob_p = eval_pcgnn(
    pcgnn_model, x_pcgnn, edge_indices_pcgnn, y, test_mask)
torch.save(best_p_state, 'best_pcgnn.pt')
print(f'\n✅ PC-GNN 최종:  PR-AUC={res_p_pr:.4f}  Macro F1={res_p_f1:.4f}')

 PC-GNN 학습
 피처: 행동 8개 + SVD 64차원 = 72차원
 엣지: R-U-R + R-S-R + R-P-R + R-Sim-R (4종) | Chooser 어텐션
 배치: Picker ratio=3  |  FocalLoss gamma=2.0
  [E 10] loss=0.2348 | val PR-AUC=0.5761 F1=0.2065 (52s)
  [E 20] loss=0.1591 | val PR-AUC=0.7703 F1=0.2160 (101s)
  [E 30] loss=0.1171 | val PR-AUC=0.8458 F1=0.7448 (146s)
  [E 40] loss=0.0939 | val PR-AUC=0.8578 F1=0.7589 (192s)
  [E 50] loss=0.0897 | val PR-AUC=0.8609 F1=0.7607 (242s)
  [E 60] loss=0.0905 | val PR-AUC=0.8614 F1=0.7610 (295s)

✅ PC-GNN 최종:  PR-AUC=0.8674  Macro F1=0.7666


In [26]:
# [12] 최종 성능 비교 리포트
W = 64
print()
print('=' * W)
print('  RGCN(Baseline) vs PC-GNN  성능 비교')
print('=' * W)

print(f'\n[1] RGCN 베이스라인 — rating_norm 1차원 / R-U-R+R-S-R / 균등 집계')
print(f'    PR-AUC  : [{bar(res_r_pr)}] {res_r_pr:.4f}')
print(f'    Macro F1: [{bar(res_r_f1)}] {res_r_f1:.4f}')

print(f'\n[2] PC-GNN — 행동피처 {len(BEHAV_COLS)}개+SVD / 4종 엣지 / Chooser+Picker+FocalLoss')
print(f'    PR-AUC  : [{bar(res_p_pr)}] {res_p_pr:.4f}')
print(f'    Macro F1: [{bar(res_p_f1)}] {res_p_f1:.4f}')

d_pr = res_p_pr - res_r_pr
d_f1 = res_p_f1 - res_r_f1
print(f'\n[3] 개선폭 (RGCN → PC-GNN)')
print(f'    PR-AUC  : {res_r_pr:.4f} → {res_p_pr:.4f}  ({d_pr:+.4f}, {d_pr/max(res_r_pr,1e-9)*100:+.1f}%)')
print(f'    Macro F1: {res_r_f1:.4f} → {res_p_f1:.4f}  ({d_f1:+.4f}, {d_f1/max(res_r_f1,1e-9)*100:+.1f}%)')

# 관계별 Attention
pcgnn_model.eval()
with torch.no_grad():
    _, attn_mean = pcgnn_model(x_pcgnn, edge_indices_pcgnn)
attn_np = attn_mean.cpu().numpy()
print(f'\n[4] 관계별 Chooser Attention (높을수록 사기 판별 기여 큼)')
for name, w, ei in zip(REL_NAMES, attn_np, edge_indices_pcgnn):
    b = '█' * int(w * 40)
    print(f'    {name:8s}: [{b:<20}] {w:.3f}  ({ei.size(1):,} edges)')

print(f'\n[5] PC-GNN — Classification Report (Test set)')
print(classification_report(yt_p, yp_p, target_names=['정상(0)','사기(1)'], digits=4))
print('=' * W)
print('✅ 완료 | 저장: best_rgcn_baseline.pt / best_pcgnn.pt')


  RGCN(Baseline) vs PC-GNN  성능 비교

[1] RGCN 베이스라인 — rating_norm 1차원 / R-U-R+R-S-R / 균등 집계
    PR-AUC  : [███████░░░░░░░░░░░░░░░░░░░░░] 0.2739
    Macro F1: [█████░░░░░░░░░░░░░░░░░░░░░░░] 0.1956

[2] PC-GNN — 행동피처 8개+SVD / 4종 엣지 / Chooser+Picker+FocalLoss
    PR-AUC  : [████████████████████████░░░░] 0.8674
    Macro F1: [█████████████████████░░░░░░░] 0.7666

[3] 개선폭 (RGCN → PC-GNN)
    PR-AUC  : 0.2739 → 0.8674  (+0.5935, +216.6%)
    Macro F1: 0.1956 → 0.7666  (+0.5710, +291.9%)

[4] 관계별 Chooser Attention (높을수록 사기 판별 기여 큼)
    R-U-R   : [█████████████████████] 0.540  (254,784 edges)
    R-S-R   : [████                ] 0.117  (481,794 edges)
    R-P-R   : [█████               ] 0.143  (724,080 edges)
    R-Sim-R : [████████            ] 0.200  (319,670 edges)

[5] PC-GNN — Classification Report (Test set)
              precision    recall  f1-score   support

       정상(0)     0.9394    0.7904    0.8585      7885
       사기(1)     0.5632    0.8413    0.6747      2533

    accuracy      

In [ ]:
# [13] 최종 결과 시각화
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, confusion_matrix

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('PC-GNN 사기 리뷰 탐지 — 최종 결과 시각화', fontsize=16, fontweight='bold')

# ── (1) 모델 성능 비교 ──────────────────────────────────────────────
ax = axes[0, 0]
x = np.arange(2)
w = 0.35
b1 = ax.bar(x - w/2, [res_r_pr, res_p_pr], w, label='PR-AUC',   color=['#90CAF9','#1565C0'], alpha=0.9)
b2 = ax.bar(x + w/2, [res_r_f1, res_p_f1], w, label='Macro F1', color=['#FFCC80','#E65100'], alpha=0.9)
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(['RGCN\n베이스라인', 'PC-GNN\n(최종)'], fontsize=11)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('모델 성능 비교 (Test Set)', fontweight='bold')
ax.legend(); ax.grid(axis='y', alpha=0.3)
d_pr = res_p_pr - res_r_pr
ax.annotate(f'PR-AUC\n+{d_pr/max(res_r_pr,1e-9)*100:.0f}%',
            xy=(0.72, res_p_pr + 0.05), fontsize=11, color='#1565C0', fontweight='bold')

# ── (2) 관계별 Attention 가중치 ─────────────────────────────────────
ax = axes[0, 1]
colors = ['#E53935', '#FB8C00', '#43A047', '#1E88E5']
edge_counts = [ei.size(1) for ei in edge_indices_pcgnn]
bars = ax.barh(REL_NAMES, attn_np, color=colors, alpha=0.88)
for bar, val, ec in zip(bars, attn_np, edge_counts):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}  ({ec:,} edges)', va='center', fontsize=10, fontweight='bold')
ax.set_xlim(0, 0.72)
ax.set_xlabel('Attention 가중치')
ax.set_title('관계별 Chooser Attention\n(모델이 학습한 관계 중요도)', fontweight='bold')
ax.grid(axis='x', alpha=0.3)
ax.invert_yaxis()

# ── (3) Precision-Recall Curve ──────────────────────────────────────
ax = axes[1, 0]
prec_p, rec_p, _ = precision_recall_curve(yt_p, yprob_p)
prec_r, rec_r, _ = precision_recall_curve(yt_r, yprob_r)
ax.plot(rec_p, prec_p, color='#1565C0', lw=2.5,
        label=f'PC-GNN  (PR-AUC = {res_p_pr:.4f})')
ax.plot(rec_r, prec_r, color='#999',   lw=1.5, linestyle='--',
        label=f'RGCN Baseline  (PR-AUC = {res_r_pr:.4f})')
ax.axhline(yt_p.mean(), color='red', linestyle=':', lw=1.2,
           label=f'Random  ({yt_p.mean():.2f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve (Test Set)', fontweight='bold')
ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
ax.legend(fontsize=9); ax.grid(alpha=0.3)

# ── (4) 혼동 행렬 ───────────────────────────────────────────────────
ax = axes[1, 1]
cm = confusion_matrix(yt_p, yp_p)
im = ax.imshow(cm, cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(['예측: 정상(0)', '예측: 사기(1)'], fontsize=10)
ax.set_yticklabels(['실제: 정상(0)', '실제: 사기(1)'], fontsize=10)
ax.set_title('Confusion Matrix — PC-GNN (Test Set)', fontweight='bold')
for i in range(2):
    for j in range(2):
        ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                fontsize=15, fontweight='bold',
                color='white' if cm[i,j] > cm.max() / 2 else 'black')

plt.tight_layout()
plt.savefig('pcgnn_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ 시각화 저장 완료: pcgnn_results.png')

In [42]:
# [14] 어블레이션 — RGCN + 파생변수 72차원
# ■ 목적: 피처 단독 기여도 측정
#   - 모델 구조·엣지는 베이스라인과 동일 (RGCN, train-only R-U-R)
#   - 피처만 1차원(rating_norm) → 72차원(행동 8 + SVD 64)으로 교체
print('='*62)
print(' RGCN + 파생변수 72차원  (어블레이션)')
print(' 피처: 행동 8개 + SVD 64차원 = 72차원')
print(' 엣지: R-U-R train-only  (베이스라인과 동일)')
print(' → 피처 단독 기여도 측정')
print('='*62)

cw_r72   = compute_class_weight('balanced', classes=np.array([0,1]), y=labels[idx_train])
crit_r72 = nn.CrossEntropyLoss(weight=torch.tensor(cw_r72, dtype=torch.float32))

rgcn72_model = BaselineRGCN(in_dim=in_dim_pcgnn, hidden_dim=64, n_rel=1, dropout=0.30)
opt_r72 = AdamW(rgcn72_model.parameters(), lr=1e-3, weight_decay=1e-4)
sch_r72 = CosineAnnealingLR(opt_r72, T_max=CFG['epochs'])

best_r72_pr, best_r72_state, no_imp72 = 0.0, None, 0
t0 = time.time()

for ep in range(1, CFG['epochs']+1):
    rgcn72_model.train(); opt_r72.zero_grad()
    logits = rgcn72_model(x_pcgnn, edge_index_rgcn, edge_type_rgcn)
    loss   = crit_r72(logits[train_mask], y[train_mask])
    loss.backward(); nn.utils.clip_grad_norm_(rgcn72_model.parameters(), 1.0); opt_r72.step()
    sch_r72.step()

    if ep % 10 == 0:
        vpr, vf1, *_ = eval_rgcn(rgcn72_model, x_pcgnn, edge_index_rgcn, edge_type_rgcn, y, val_mask)
        print(f'  [E{ep:3d}] loss={float(loss):.4f} | val PR-AUC={vpr:.4f} F1={vf1:.4f} ({time.time()-t0:.0f}s)')
        if vpr > best_r72_pr:
            best_r72_pr = vpr
            best_r72_state = {k: v.clone() for k,v in rgcn72_model.state_dict().items()}
            no_imp72 = 0
        else:
            no_imp72 += 1
            if no_imp72 >= CFG['patience'] // 2:
                print('  Early stopping'); break

rgcn72_model.load_state_dict(best_r72_state)
res_r72_pr, res_r72_f1, yt_r72, yp_r72, yprob_r72 = eval_rgcn(
    rgcn72_model, x_pcgnn, edge_index_rgcn, edge_type_rgcn, y, test_mask)
print(f'\n✅ RGCN+72차원 최종:  PR-AUC={res_r72_pr:.4f}  Macro F1={res_r72_f1:.4f}')


 RGCN + 파생변수 72차원  (어블레이션)
 피처: 행동 8개 + SVD 64차원 = 72차원
 엣지: R-U-R train-only  (베이스라인과 동일)
 → 피처 단독 기여도 측정
  [E 10] loss=0.5740 | val PR-AUC=0.6057 F1=0.2262 (3s)
  [E 20] loss=0.4473 | val PR-AUC=0.6012 F1=0.2079 (5s)
  [E 30] loss=0.3570 | val PR-AUC=0.5957 F1=0.2065 (8s)
  [E 40] loss=0.3060 | val PR-AUC=0.5968 F1=0.2065 (10s)
  [E 50] loss=0.2866 | val PR-AUC=0.5978 F1=0.2065 (13s)
  [E 60] loss=0.2817 | val PR-AUC=0.5980 F1=0.2065 (18s)
  Early stopping

✅ RGCN+72차원 최종:  PR-AUC=0.6088  Macro F1=0.2106


In [43]:
# [15] 단계별 개선 분석 — 모델 기반 인사이트
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 단계 1 │ RGCN + rating_norm 1차원  (순수 베이스라인)
# 단계 2 │ RGCN + 파생변수 72차원   (피처만 교체, 모델·엣지 동일)
# 단계 3 │ PC-GNN + 파생변수 72차원 (모델·엣지·학습 전략 전체 교체)
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

W = 72
sep = '━' * W

# ─── 단계별 수치 정리 ─────────────────────────────────────
stages = [
    ('단계 1', 'RGCN  + 1차원 피처',  res_r_pr,   res_r_f1),
    ('단계 2', 'RGCN  + 72차원 파생', res_r72_pr, res_r72_f1),
    ('단계 3', 'PC-GNN + 72차원 파생', res_p_pr,   res_p_f1),
]

d12_pr = res_r72_pr - res_r_pr;   d12_f1 = res_r72_f1 - res_r_f1
d23_pr = res_p_pr  - res_r72_pr;  d23_f1 = res_p_f1  - res_r72_f1
d13_pr = res_p_pr  - res_r_pr;    d13_f1 = res_p_f1  - res_r_f1

feat_share_pr  = d12_pr / d13_pr * 100 if d13_pr > 0 else 0
model_share_pr = d23_pr / d13_pr * 100 if d13_pr > 0 else 0
feat_share_f1  = d12_f1 / d13_f1 * 100 if d13_f1 > 0 else 0
model_share_f1 = d23_f1 / d13_f1 * 100 if d13_f1 > 0 else 0

print()
print('=' * W)
print('  단계별 개선 분석 — 모델 기반 인사이트')
print('=' * W)

# ─── 표: 단계별 결과 ──────────────────────────────────────
print(f'\n{"":4} {"단계":8} {"설명":30} {"PR-AUC":>9} {"Macro F1":>9}')
print('─' * W)
for tag, desc, pr, f1 in stages:
    bar_p = '█' * int(pr * 24)
    print(f'  {tag}  {desc:<30} {pr:>9.4f} {f1:>9.4f}')
print('─' * W)

# ─── 구간별 개선폭 ────────────────────────────────────────
print(f"""
  ▲ 1→2  피처 교체 효과 (RGCN 1차원 → RGCN 72차원)
      PR-AUC  : {res_r_pr:.4f} → {res_r72_pr:.4f}  ({d12_pr:+.4f})
      Macro F1: {res_r_f1:.4f} → {res_r72_f1:.4f}  ({d12_f1:+.4f})

  ▲ 2→3  모델 교체 효과 (RGCN 72차원 → PC-GNN 72차원)
      PR-AUC  : {res_r72_pr:.4f} → {res_p_pr:.4f}  ({d23_pr:+.4f})
      Macro F1: {res_r72_f1:.4f} → {res_p_f1:.4f}  ({d23_f1:+.4f})

  ▲ 1→3  전체 개선폭 (베이스라인 → 최종)
      PR-AUC  : {res_r_pr:.4f} → {res_p_pr:.4f}  ({d13_pr:+.4f}, +{d13_pr/max(res_r_pr,1e-9)*100:.1f}%)
      Macro F1: {res_r_f1:.4f} → {res_p_f1:.4f}  ({d13_f1:+.4f}, +{d13_f1/max(res_r_f1,1e-9)*100:.1f}%)
""")

print(sep)
print('  전체 개선분 기여 비율')
print(sep)
print(f'  PR-AUC  기준  │  피처 교체(1→2): {feat_share_pr:5.1f}%  │  모델 교체(2→3): {model_share_pr:5.1f}%')
print(f'  Macro F1 기준 │  피처 교체(1→2): {feat_share_f1:5.1f}%  │  모델 교체(2→3): {model_share_f1:5.1f}%')

print(f"""
{sep}
  인사이트 해석
{sep}

  [인사이트 1] 피처 교체(1→2)에서 얼마나 올랐나?
    · RGCN 모델 구조와 엣지 구성을 그대로 유지한 채
      rating_norm 1차원 → 행동 8개 + TF-IDF SVD 64차원으로만 교체
    · val/test 노드는 여전히 그래프에서 고립 → 그래프 구조 정보 없음
    · 그럼에도 PR-AUC {d12_pr:+.4f} 상승:
      행동 피처(버스트/활동 기간/집중도)와 텍스트 피처가
      개별 노드 수준에서 이미 강력한 사기 신호를 담고 있음을 입증

  [인사이트 2] 모델 교체(2→3)에서 얼마나 더 올랐나?
    · 피처를 72차원으로 고정한 채 RGCN → PC-GNN으로 전환
      (4종 이질 엣지 + Chooser 어텐션 + Picker + FocalLoss)
    · PR-AUC {d23_pr:+.4f} 추가 상승:
      ① R-S-R·R-P-R·R-Sim-R 엣지로 val/test 노드가 그래프에 연결됨
         → 이웃 사기 신호가 전파되어 개별 피처 한계 극복
      ② Chooser 어텐션이 위장 정상 이웃 억제
         (R-U-R Attention 0.540 — 동일 유저 관계가 가장 강한 신호)
      ③ Picker가 사기 전량 + 정상 3배 배치 구성
         → 불균형 환경에서 사기 그라디언트 신호 3배 이상 증폭
      ④ FocalLoss가 쉬운 정상 샘플 손실 자동 감쇠

  [인사이트 3] 두 요인의 구조적 의미
    · 피처 기여({feat_share_pr:.0f}%): EDA에서 발굴한 버스트/행동 패턴이
      GNN 없이도 판별력 있음 → EDA 품질이 모델 성능의 토대
    · 모델 기여({model_share_pr:.0f}%): 그래프 구조 + Picker/Chooser/FocalLoss가
      피처만으로 포착 못한 조직적 공모 패턴을 추가로 포착
    · 결론: 피처 설계(EDA)와 GNN 구조는 상호 보완적 — 둘 중 하나만으로는
      현재 성능 달성 불가
""")
print('=' * W)
print('✅ 단계별 인사이트 출력 완료')



  단계별 개선 분석 — 모델 기반 인사이트

     단계       설명                                PR-AUC  Macro F1
────────────────────────────────────────────────────────────────────────
  단계 1  RGCN  + 1차원 피처                    0.2739    0.1956
  단계 2  RGCN  + 72차원 파생                   0.6088    0.2106
  단계 3  PC-GNN + 72차원 파생                  0.8674    0.7666
────────────────────────────────────────────────────────────────────────

  ▲ 1→2  피처 교체 효과 (RGCN 1차원 → RGCN 72차원)
      PR-AUC  : 0.2739 → 0.6088  (+0.3349)
      Macro F1: 0.1956 → 0.2106  (+0.0150)

  ▲ 2→3  모델 교체 효과 (RGCN 72차원 → PC-GNN 72차원)
      PR-AUC  : 0.6088 → 0.8674  (+0.2586)
      Macro F1: 0.2106 → 0.7666  (+0.5560)

  ▲ 1→3  전체 개선폭 (베이스라인 → 최종)
      PR-AUC  : 0.2739 → 0.8674  (+0.5935, +216.6%)
      Macro F1: 0.1956 → 0.7666  (+0.5710, +291.9%)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  전체 개선분 기여 비율
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  PR-AUC  기준  │  피처 교체(1→2):  